In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# AdaBoost

In [5]:
# Класс базовой модели - дерева глубины 1
class DecisionStump:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1 #направление разбиения
        self.alpha = None

    def predict(self, X):
        n_samples = X.shape[0]
        feature = X[:, self.feature_index]
        predictions = np.ones(n_samples)
        if self.polarity == 1:
            predictions[feature < self.threshold] = -1
        else:
            predictions[feature > self.threshold] = -1
        return predictions

In [6]:
# AdaBoost алгоритм
class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.stumps = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Инициализация весов
        w = np.full(n_samples, 1 / n_samples)

        for _ in range(self.n_estimators):
            stump = DecisionStump()
            min_error = float("inf")

            # Поиск наилучшего пенька
            for feature_i in range(n_features):#просто перебираем все подряд
                feature_values = X[:, feature_i]
                thresholds = np.unique(feature_values)
                for threshold in thresholds:
                    for polarity in [1, -1]:
                        predictions = np.ones(n_samples)
                        if polarity == 1:
                            predictions[feature_values < threshold] = -1
                        else:
                            predictions[feature_values > threshold] = -1

                        error = np.sum(w[y != predictions])#ошибка - сумма весов неправильно классифицируемых обьктов

                        if error < min_error:
                            stump.feature_index = feature_i
                            stump.threshold = threshold
                            stump.polarity = polarity
                            min_error = error

            # Вычисление веса (альфа) модели
            EPS = 1e-10  # для избежания деления на 0
            stump.alpha = 0.5 * np.log((1 - min_error + EPS) / (min_error + EPS))#экспоненциальная функция потерь
            predictions = stump.predict(X)
            w *= np.exp(-stump.alpha * y * predictions) #если обьект правильно классифицировался вес уменьшился
            w /= np.sum(w) #нормализация

            self.stumps.append(stump)

    def predict(self, X):
        stump_preds = [stump.alpha * stump.predict(X) for stump in self.stumps]
        return np.sign(np.sum(stump_preds, axis=0))


In [8]:
from sklearn.metrics import roc_auc_score
#тест на синтетике
def generate_data(n=200):
    np.random.seed(42)
    X = np.random.randn(n, 2)
    y = np.where(X[:, 0] * X[:, 1] > 0, 1, -1)
    return X, y

X, y = generate_data()

model = AdaBoost(n_estimators=10)
model.fit(X, y)

y_score = np.sum([stump.alpha * stump.predict(X) for stump in model.stumps], axis=0)
roc_auc = roc_auc_score((y == 1).astype(int), y_score)
print("AdaBoost, ROC AUC:", round(roc_auc, 4))


AdaBoost, ROC AUC: 0.6772


In [9]:
# сравнение с sklearn
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

sk_model = AdaBoostClassifier(
    base_estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=10
)
sk_model.fit(X, (y == 1).astype(int))
sk_y_score = sk_model.decision_function(X)
sk_roc_auc = roc_auc_score((y == 1).astype(int), sk_y_score)
print("Sklearn AdaBoost, ROC AUC:", round(sk_roc_auc, 4))


Sklearn AdaBoost, ROC AUC: 0.6974


C:\Users\kamil\anaconda3\Lib\site-packages\sklearn\ensemble\_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


In [10]:
#Реальные данные
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X_real, y_real = data.data, data.target
y_real = np.where(y_real == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.3, random_state=42)

model_real = AdaBoost(n_estimators=30)
model_real.fit(X_train, y_train)
y_score_real = np.sum([stump.alpha * stump.predict(X_test) for stump in model_real.stumps], axis=0)
roc_auc_real = roc_auc_score((y_test == 1).astype(int), y_score_real)
print("Наша реализация на реальных данных, ROC AUC:", round(roc_auc_real, 4))

Наша реализация на реальных данных, ROC AUC: 0.9972


# GradientBoosting

In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

In [15]:
class DecisionStump:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.left_value = None
        self.right_value = None

    def fit(self, X, y):
        m, n = X.shape
        min_error = float("inf")

        for feature_i in range(n):
            thresholds = np.unique(X[:, feature_i])
            for threshold in thresholds:
                left_mask = X[:, feature_i] <= threshold
                right_mask = ~left_mask

                left_value = np.mean(y[left_mask]) if np.any(left_mask) else 0 #среднее значение таргета - предсказние
                right_value = np.mean(y[right_mask]) if np.any(right_mask) else 0

                pred = np.where(left_mask, left_value, right_value)
                error = np.mean((y - pred) ** 2) # по факту просто MSE

                if error < min_error:
                    min_error = error
                    self.feature_index = feature_i
                    self.threshold = threshold
                    self.left_value = left_value
                    self.right_value = right_value

    def predict(self, X):
        feature = X[:, self.feature_index]
        return np.where(feature <= self.threshold, self.left_value, self.right_value)

In [16]:
class GradientBoosting:
    def __init__(self, n_estimators=100, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.models = []
        self.initial_pred = 0

    def fit(self, X, y):
        y = (y == 1).astype(np.float64)
        self.initial_pred = np.log(np.mean(y) / (1 - np.mean(y))) #логарифм шансов - начальное предсказание для всех 
        F = np.full(y.shape, self.initial_pred)

        for _ in range(self.n_estimators):
            p = 1 / (1 + np.exp(-F))   # вероятность класса 1 (сигмоида)
            residual = y - p           # псевдо-остатки (градиент функции потерь log-loss)

            stump = DecisionStump()
            stump.fit(X, residual) #работаем с градиентом функции потерь а не с весами как  врпдыдущем алгоритмне
            self.models.append(stump)

            update = stump.predict(X)
            F += self.learning_rate * update

    def predict_proba(self, X):
        F = np.full(X.shape[0], self.initial_pred)
        for model in self.models:
            F += self.learning_rate * model.predict(X)
        return 1 / (1 + np.exp(-F))

    def predict(self, X):
        return np.where(self.predict_proba(X) >= 0.5, 1, -1)


In [17]:
def generate_data(n=200):
    np.random.seed(42)
    X = np.random.randn(n, 2)
    y = np.where(X[:, 0] * X[:, 1] > 0, 1, -1)
    return X, y

X, y = generate_data()
model = GradientBoosting(n_estimators=50, learning_rate=0.1)
model.fit(X, y)
proba = model.predict_proba(X)
roc_auc = roc_auc_score((y == 1).astype(int), proba)
print("Наша реализация GradientBoosting (синтетика), ROC AUC:", round(roc_auc, 4))

sk_model = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=1)
sk_model.fit(X, (y == 1).astype(int))
sk_proba = sk_model.predict_proba(X)[:, 1]
sk_roc_auc = roc_auc_score((y == 1).astype(int), sk_proba)
print("Sklearn GradientBoosting (синтетика), ROC AUC:", round(sk_roc_auc, 4))

Наша реализация GradientBoosting (синтетика), ROC AUC: 0.5986
Sklearn GradientBoosting (синтетика), ROC AUC: 0.6978


In [18]:
data = load_breast_cancer()
X_real, y_real = data.data, data.target
y_real = np.where(y_real == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.3, random_state=42)

model_real = GradientBoosting(n_estimators=100, learning_rate=0.1)
model_real.fit(X_train, y_train)
proba_real = model_real.predict_proba(X_test)
roc_auc_real = roc_auc_score((y_test == 1).astype(int), proba_real)
print("Наша реализация GradientBoosting (реальные данные), ROC AUC:", round(roc_auc_real, 4))

sk_model_real = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=1)
sk_model_real.fit(X_train, (y_train == 1).astype(int))
sk_proba_real = sk_model_real.predict_proba(X_test)[:, 1]
sk_roc_auc_real = roc_auc_score((y_test == 1).astype(int), sk_proba_real)
print("Sklearn GradientBoosting (реальные данные), ROC AUC:", round(sk_roc_auc_real, 4))

Наша реализация GradientBoosting (реальные данные), ROC AUC: 0.9902
Sklearn GradientBoosting (реальные данные), ROC AUC: 0.9951
